<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [122]:
!pip3 install beautifulsoup4
!pip3 install requests

In [123]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [124]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    extracted = [data_time.strip() for data_time in list (table_cells.strings)]
    if len(extracted) == 1:
        return extracted[0], 'N/A'

    return extracted[0], extracted[1]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out = ''.join([booster_version for i, booster_version in enumerate( table_cells.strings) if i%2==0][:-1])
    return out if out else table_cells.get_text(strip=True)

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    return list(table_cells.stripped_strings)[0]


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if "kg" in mass:
        return mass[:mass.find("kg")+2]
    return "0 kg"


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()

    column_name = ' '.join(row.stripped_strings)
    
    if column_name and not column_name.strip().isdigit():
        return column_name
    return None





To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [125]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [126]:
# use requests.get() method with the provided static_url
response=requests.get(static_url)

# assign the response to a object
response.status_code

200

Create a `BeautifulSoup` object from the HTML `response`


In [127]:
# Use BeautifulSoup() to create a BeautifulSoup object from a response text content
soup = BeautifulSoup(response.text, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [128]:
# Use soup.title attribute
soup.find('title')

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [129]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
html_tables = soup.find_all('table', class_='wikitable')


Starting from the third table is our target table contains the actual launch records.


In [132]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table)

<table class="wikitable plainrowheaders collapsible" style="width: 100%;">
<tbody><tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a><sup class="reference" id="cite_ref-booster_11-2"><a href="#cite_note-booster-11"><span class="cite-bracket">[</span>b<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-2"><a href="#cite_note-Dragon-12"><span class="cite-bracket">[</span>c<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9

You should able to see the columns names embedded in the table header elements `<th>` as follows:


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [144]:
column_names = [extract_column_from_header(th) for th in first_launch_table.find_all('th')]
column_names = [name for name in column_names if name]  # Remove None values

# Apply find_all() function with `th` element on first_launch_table
# Iterate each th element and apply the provided extract_column_from_header() to get a column name
# Append the Non-empty column name (`if name is not None and len(name) > 0`) into a list called column_names


Check the extracted column names


In [145]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


In [146]:
if "Date and Time ( )" in column_names:
    column_names[column_names.index("Date and Time ( )")] = "Date"
    column_names.insert(column_names.index("Date") + 1, "Time")

## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [147]:
expected_columns = [
    'Flight No.', 'Date', 'Time', 'Version Booster', 'Launch site',
    'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing'
]

In [148]:
launch_dict = {key: [] for key in expected_columns}

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [138]:

#Extract each table 
for table in soup.find_all('table', class_="wikitable plainrowheaders collapsible"):

   # get table row 
    for rows in table.find_all("tr"):

        #check to see if first table heading is as number corresponding to launch a number 
        if rows.th and rows.th.string:
                flight_number = rows.th.string.strip()
                if flight_number.isdigit():

        #get table element 
                    row = rows.find_all('td')

        #if it is number save cells in a dictonary 
        # date.time list
                    date, time = date_time(row[0])
        # Date value
                 #   date = datatimelist[0].strip(',')
        # Time value
                 #   time = datatimelist[1] if len(datatimelist) > 1 else "N/A"
        # Booster version
                    bv = booster_version(row[1])
        # Launch Site
                    launch_site = row[2].a.string if row[2].a else 'N/A'
        # Payload
                    payload = row[3].a.string if row[3].a else 'N/A'
        # Payload Mass
                    payload_mass = get_mass(row[4])
        # Orbit
                    orbit = row[5].a.string if row[5].a else 'N/A'
        # Customer
                    customer = row[6].a.string if row[6].a else 'N/A'
        # Launch outcome
                    launch_outcome = list(row[7].strings)[0] if row[7].strings else 'N/A'
        # Booster landing
                    booster_landing = landing_status(row[8])
     

In [139]:
column_names

['Flight No.',
 'Date and time ( )',
 'Launch site',
 'Payload',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launch outcome']

In [141]:
  # Append data to dictionary
launch_dict['Flight No.'].append(flight_number)
launch_dict['Date'].append(date)
launch_dict['Time'].append(time)
launch_dict['Version Booster'].append(bv)
launch_dict['Launch site'].append(launch_site)
launch_dict['Payload'].append(payload)
launch_dict['Payload mass'].append(payload_mass)
launch_dict['Orbit'].append(orbit)
launch_dict['Customer'].append(customer)
launch_dict['Launch outcome'].append(launch_outcome)
launch_dict['Booster landing'].append(booster_landing)


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [142]:
df= pd.DataFrame(launch_dict)

In [143]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Flight No.       2 non-null      object
 1   Date             2 non-null      object
 2   Time             2 non-null      object
 3   Version Booster  2 non-null      object
 4   Launch site      2 non-null      object
 5   Payload          2 non-null      object
 6   Payload mass     2 non-null      object
 7   Orbit            2 non-null      object
 8   Customer         2 non-null      object
 9   Launch outcome   2 non-null      object
 10  Booster landing  2 non-null      object
dtypes: object(11)
memory usage: 304.0+ bytes


In [149]:
df.shape

(2, 11)

We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
